# 02 — Análisis: ¿qué juegos valen su precio?

Consultas SQL sobre la base generada en el notebook anterior. Cada sección responde una pregunta concreta de negocio.

**Input:** `data/clean/steam.db`  
**Output:** gráficos en `img/`

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

DB_PATH = '../data/clean/steam.db'

def query(sql):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn)

## Vista general del catálogo

In [ ]:
resumen = query("""
    SELECT
        COUNT(*) AS total_juegos,
        SUM(CASE WHEN price = 0 THEN 1 ELSE 0 END) AS gratuitos,
        SUM(CASE WHEN price > 0 THEN 1 ELSE 0 END) AS de_pago,
        ROUND(AVG(CASE WHEN price > 0 THEN price END), 2) AS precio_promedio,
        ROUND(AVG(CASE WHEN review_score IS NOT NULL THEN review_score END), 1) AS review_promedio
    FROM games
""")
resumen

## Pregunta 1: ¿qué géneros tienen mejor review score?

Solo géneros con al menos 50 juegos y 100+ reviews para evitar sesgos de muestra pequeña.

In [ ]:
df_generos = query("""
    SELECT
        genres,
        COUNT(*) AS total_juegos,
        ROUND(AVG(review_score), 1) AS review_promedio,
        ROUND(AVG(price), 2) AS precio_promedio,
        ROUND(AVG(average_playtime), 0) AS playtime_promedio
    FROM games
    WHERE review_score IS NOT NULL
      AND total_ratings >= 100
      AND genres NOT LIKE '%,%'
    GROUP BY genres
    HAVING total_juegos >= 50
    ORDER BY review_promedio DESC
    LIMIT 15
""")

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(df_generos['genres'], df_generos['review_promedio'],
               color='#5b8db8', alpha=0.85)
ax.set_xlabel('Review score promedio (%)')
ax.set_title('Top géneros por review score — Steam')
ax.set_xlim(50, 100)
for bar, val in zip(bars, df_generos['review_promedio']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../img/02_review_por_genero.png', dpi=150)
plt.show()
df_generos

## Pregunta 2: ¿cuáles son los juegos con más horas por dólar?

Filtramos juegos con precio entre $1 y $60, review score ≥ 70% y al menos 1 hora de playtime registrada.

In [ ]:
df_valor = query("""
    SELECT
        name,
        ROUND(price, 2) AS price,
        average_playtime,
        ROUND(review_score, 1) AS review_score,
        ROUND(horas_por_dolar, 1) AS horas_por_dolar,
        genres
    FROM games
    WHERE price BETWEEN 1 AND 60
      AND average_playtime >= 60
      AND review_score >= 70
      AND horas_por_dolar IS NOT NULL
    ORDER BY horas_por_dolar DESC
    LIMIT 20
""")

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(df_valor['name'].str[:40], df_valor['horas_por_dolar'],
        color='#4a9e6b', alpha=0.85)
ax.set_xlabel('Horas de juego por dólar')
ax.set_title('Top 20 juegos con más horas por dólar (review ≥ 70%)')
plt.tight_layout()
plt.savefig('../img/02_horas_por_dolar.png', dpi=150)
plt.show()
df_valor

## Pregunta 3: ¿hay correlación entre precio y review score?

In [ ]:
df_scatter = query("""
    SELECT price, review_score, total_ratings, average_playtime, name
    FROM games
    WHERE price BETWEEN 0 AND 60
      AND review_score IS NOT NULL
      AND total_ratings >= 50
""")

correlacion = df_scatter['price'].corr(df_scatter['review_score'])
print(f'Correlación precio vs review score: {correlacion:.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(
    df_scatter['price'], df_scatter['review_score'],
    alpha=0.3, s=15,
    c=df_scatter['total_ratings'].clip(0, 10000),
    cmap='viridis'
)
plt.colorbar(sc, ax=ax, label='Total ratings')
ax.set_xlabel('Precio (USD)')
ax.set_ylabel('Review score (%)')
ax.set_title(f'Precio vs review score  (r = {correlacion:.2f})')
plt.tight_layout()
plt.savefig('../img/02_precio_vs_review.png', dpi=150)
plt.show()

## Pregunta 4: ¿los juegos gratuitos tienen peores reviews?

In [ ]:
df_f2p = query("""
    SELECT
        CASE WHEN price = 0 THEN 'Gratuito' ELSE 'De pago' END AS tipo,
        ROUND(AVG(review_score), 1) AS review_promedio,
        ROUND(AVG(average_playtime), 0) AS playtime_promedio,
        COUNT(*) AS total
    FROM games
    WHERE review_score IS NOT NULL
      AND total_ratings >= 50
    GROUP BY tipo
""")
print(df_f2p.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, label in zip(axes,
    ['review_promedio', 'playtime_promedio'],
    ['Review score promedio (%)', 'Playtime promedio (min)']):
    ax.bar(df_f2p['tipo'], df_f2p[col],
           color=['#5b8db8', '#e07b39'], alpha=0.85)
    ax.set_ylabel(label)
    for i, v in enumerate(df_f2p[col]):
        ax.text(i, v + 0.5, str(v), ha='center', fontsize=11)
plt.suptitle('Gratuito vs de pago: reviews y playtime', fontsize=13)
plt.tight_layout()
plt.savefig('../img/02_f2p_vs_pago.png', dpi=150)
plt.show()

## Pregunta 5: ¿qué año tuvo los mejores lanzamientos?

In [ ]:
df_anio = query("""
    SELECT
        release_year,
        COUNT(*) AS lanzamientos,
        ROUND(AVG(review_score), 1) AS review_promedio
    FROM games
    WHERE release_year BETWEEN 2000 AND 2019
      AND review_score IS NOT NULL
      AND total_ratings >= 50
    GROUP BY release_year
    ORDER BY release_year
""")

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

ax1.bar(df_anio['release_year'], df_anio['lanzamientos'],
        color='#5b8db8', alpha=0.6, label='Lanzamientos')
ax2.plot(df_anio['release_year'], df_anio['review_promedio'],
         color='#e07b39', marker='o', linewidth=2, label='Review promedio')

ax1.set_xlabel('Año')
ax1.set_ylabel('Cantidad de lanzamientos', color='#5b8db8')
ax2.set_ylabel('Review score promedio (%)', color='#e07b39')
ax2.set_ylim(50, 100)
ax1.set_title('Lanzamientos y review score por año — Steam')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('../img/02_lanzamientos_por_anio.png', dpi=150)
plt.show()

## Resumen de hallazgos

*(Completar con los valores reales al ejecutar el notebook)*

- **Mejor género por reviews:** TBD
- **Juego con más horas por dólar:** TBD
- **Correlación precio vs review:** TBD — lo que indica que precio y calidad percibida no están relacionados linealmente
- **Gratuito vs pago:** TBD
- **Año con mejor review promedio:** TBD